In [1]:
import os
import subprocess
import pandas as pd
import numpy as np
import json
import glob
import requests

In [2]:
def setup_datasets():
    #Define the target directory for the datasets
    raw_data_path = os.path.join("..", "data", "raw")
    os.makedirs(raw_data_path, exist_ok=True)

    # 1. Clone AVeriTeC (Fact-checking Claims—used by Claimify)
    averitec_dir = os.path.join(raw_data_path, "averitec")
    if not os.path.exists(averitec_dir):
        print("Cloning AVeriTeC into data/raw/...")
        subprocess.run(["git", "clone", "https://github.com/MichSchli/AVeriTeC.git", averitec_dir])

    # 2. Clone SemEval-2020 Task 11 (Span-level Propaganda Labels)
    semeval_file = os.path.join(raw_data_path, "semeval-dataset.tgz")
    if not os.path.exists(semeval_file):
        print("Downloading SemEval-2020 Task 11 dataset...")
        # -L follows redirects, -o specifies output path
        subprocess.run(["curl", "-L", "-o", semeval_file, "https://zenodo.org/record/3952415/files/datasets-v2.tgz?download=1"])
        print("Download complete.")

    # 3. Clone HQP (High-Quality Propaganda Social Media Dataset)
    hqp_dir = os.path.join(raw_data_path, "hqp")
    if not os.path.exists(hqp_dir):
        print("Cloning HQP Dataset Repo into data/raw/...")
        subprocess.run(["git", "clone", "https://github.com/abdumaa/HiQualProp.git", hqp_dir])

    print("\nDatasets initialized in data/raw/.")

if __name__ == "__main__":
    setup_datasets()

Cloning AVeriTeC into data/raw/...


Cloning into '../data/raw/averitec'...


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   263  100   263    0     0    372      0 --:--:-- --:--:-- --:--:--   372
100 1115k  100 1115k    0     0   563k      0  0:00:01  0:00:01 --:--:-- 1195k
Cloning into '../data/raw/hqp'...


Download complete.
Cloning HQP Dataset Repo into data/raw/...

Datasets initialized in data/raw/.


In [6]:
#Load AVeriTeC data into a dataframe
def load_averitec(file_path='../data/raw/averitec/data/dev.json'):
    with open(file_path, 'r') as f:
        data = json.load(f)

    # Flattening the JSON structure
    df = pd.json_normalize(data)
    return df

averitec = load_averitec()
averitec.head()

,claim,required_reannotation,label,justification,claim_date,speaker,original_claim_url,fact_checking_article,reporting_source,location_ISO_code,claim_types,fact_checking_strategies,questions,cached_original_claim_url
0,"In a letter to Steve Jobs, Sean Connery refuse...",False,Refuted,The answer and sources show that the claim was...,31-10-2020,NaN,NaN,https://web.archive.org/web/20201130144023/htt...,Facebook,NaN,[Event/Property Claim],[Written Evidence],[{'question': 'Where was the claim first publi...,NaN
1,Trump Administration claimed songwriter Billie...,False,Refuted,Seems that the Wzshington post accused the sin...,31-10-2020,NaN,NaN,https://web.archive.org/web/20201103001419/htt...,Instagram,US,"[Position Statement, Event/Property Claim]",[Written Evidence],[{'question': 'Has the Trump administration vo...,NaN
2,Due to Imran Khan's criticism of Macron's comm...,False,Refuted,The tweet was not the official government page...,31-10-2020,Consulate General Of Pakistan France,https://web.archive.org/web/20201113115127/htt...,https://web.archive.org/web/20210629013122/htt...,Twitter,FR,"[Causal Claim, Event/Property Claim]",[Written Evidence],[{'question': 'How did Macron criticise Islam?...,https://web.archive.org/web/20201113115127/htt...
3,UNESCO declared Nadar community as the most an...,False,Refuted,This claim is refuted. According to the QA pai...,31-10-2020,Kumar Shankar,NaN,https://web.archive.org/web/20210225110220/htt...,Facebook,IN,[Event/Property Claim],[Written Evidence],"[{'question': 'What is Nadar?', 'answers': [{'...",NaN
4,Republican Matt Gaetz was part of a company th...,True,Refuted,The company was sold in 2004 and the law suit ...,31-10-2020,,NaN,https://web.archive.org/web/20210713185816/htt...,Facebook,US,"[Numerical Claim, Event/Property Claim]","[Written Evidence, Numerical Comparison]",[{'question': 'Did Matt Gaetz work for Chemed ...,NaN


In [7]:
#Load HQP data into a dataframe
def load_hqp_labels(file_path='../data/raw/hqp/Data/HiQualProp/HiQualProp.csv'):
    # This loads the IDs and the 'propaganda_strategy' labels
    df = pd.read_csv(file_path)
    return df

#id column shows Tweet ID but we have to run with Twitter API to get actual Tweet text
hqp = load_hqp_labels()
hqp.head()

,id,labels,strategy,labels_weak1,labels_weak3
0,1372969669389393932,0,general discourse,0,0
1,1521594471279996928,0,general discourse,0,0
2,1385896329071632386,0,general discourse,0,1
3,1403386698835365888,0,general discourse,0,1
4,1509240723362795531,0,general discourse,0,1


In [10]:
#Load TGZ data
tgz_path = "../data/raw/semeval-dataset.tgz"
extract_dir = "../data/raw/semeval2020_data"

print(f"File exists: {os.path.exists(tgz_path)} | Path: {tgz_path}")

#Make a target directory
os.makedirs(extract_dir, exist_ok=True)

#Extract with tar
!tar -xvzf "$tgz_path" -C "$extract_dir"
print(f"\nExtraction complete! Files are in: {extract_dir}")

File exists: True | Path: ../data/raw/semeval-dataset.tgz
x datasets/
x datasets/README.md
x datasets/train-labels-task1-span-identification/
x datasets/train-labels-task1-span-identification/article728972961.task1-SI.labels
x datasets/train-labels-task1-span-identification/article111111136.task1-SI.labels
x datasets/train-labels-task1-span-identification/article790667730.task1-SI.labels
x datasets/train-labels-task1-span-identification/article770156851.task1-SI.labels
x datasets/train-labels-task1-span-identification/article786250729.task1-SI.labels
x datasets/train-labels-task1-span-identification/article766632016.task1-SI.labels
x datasets/train-labels-task1-span-identification/article786344683.task1-SI.labels
x datasets/train-labels-task1-span-identification/article999000880.task1-SI.labels
x datasets/train-labels-task1-span-identification/article999000147.task1-SI.labels
x datasets/train-labels-task1-span-identification/article696694316.task1-SI.labels
x datasets/train-labels-task

In [ ]:
base_dir = "semeval2020_data"

si_files = []
tc_files = []

for path in glob.glob(os.path.join(base_dir, "**", "*.labels"), recursive=True):
    fname = os.path.basename(path)
    if "task1-SI" in fname:
        si_files.append(path)
    elif "task2-TC" in fname:
        tc_files.append(path)

# Load SI (no technique column)
df_si_list = []
for path in si_files:
    df = pd.read_csv(
        path, sep="\t", header=None,
        names=["article_id", "start_char", "end_char"]
    )
    df["source_file"] = os.path.basename(path)
    df_si_list.append(df)
df_si = pd.concat(df_si_list, ignore_index=True)

# Load TC (with technique column second)
df_tc_list = []
for path in tc_files:
    df = pd.read_csv(
        path, sep="\t", header=None,
        names=["article_id", "technique", "start_char", "end_char"]
    )
    df["source_file"] = os.path.basename(path)
    df_tc_list.append(df)
df_tc = pd.concat(df_tc_list, ignore_index=True)
df_tc

In [ ]:
text_files = glob.glob(os.path.join(base_dir, "**", "*.txt"), recursive=True)
print("Found text files:", text_files[:10])  # show a sample

articles = []

for path in text_files:
    fname = os.path.basename(path)          # e.g. "article763280007.txt"
    article_id_str = os.path.splitext(fname)[0]  # "article763280007"
    # Strip the "article" prefix if present
    if article_id_str.startswith("article"):
        article_id_str = article_id_str[len("article"):]  # "763280007"
    article_id = int(article_id_str)          # numeric to match labels

    with open(path, encoding="utf-8") as f:
        text = f.read()

    articles.append({"article_id": article_id, "text": text})

df_articles = pd.DataFrame(articles)
df_articles.head()

In [ ]:
sem_eval = df_tc.merge(df_articles[["article_id", "text"]], on="article_id", how="left")
sem_eval.head()

In [ ]:
# Convert comma-separated technique strings to lists
sem_eval['technique'] = sem_eval['technique'].str.split(',')

# Clean up whitespace from each technique name
sem_eval['technique'] = sem_eval['technique'].apply(lambda x: [t.strip() for t in x])
sem_eval

In [ ]:
#Only show relevant text being evaluated for each row (using span on text)
sem_eval['span_text'] = sem_eval.apply(lambda row: row['text'][row['start_char']:row['end_char']], axis=1)
sem_eval